# Tile-wise PNG quantization benchmark for gsplat `PngCompression`

Does `PngCompression(tile_size=..., bits=...)` (one min/max per channel per `B x B` block of the
PLAS-sorted grid) beat the current scheme (one min/max per channel for the whole scene) in
rate-distortion? Scenes: MipNeRF360 **garden** and **bicycle**, trained and evaluated with the exact
settings of `examples/benchmarks/compression/mcmc.sh` (MCMC, 1M Gaussians, LPIPS VGG).

**Kaggle settings:** Accelerator *GPU T4* (GPU 0 is used), Internet *on*.

**Resuming:** every step skips work whose outputs exist. To continue in a new session, add this
notebook's previous output as an input; step 1 copies `results/`, `tilequant/` and `wheels/` back into
`/kaggle/working` (checkpoints, cached sort/k-means, built wheel, results CSV).

| Step | What |
|---|---|
| 1 | config, restore previous output |
| 2 | environment report |
| 3 | install gsplat from the fork branch (`MAX_JOBS=2`), example deps |
| 4 | MipNeRF360 data (Kaggle input if present, else only the needed files of `360_v2.zip`) |
| 5 | train MCMC checkpoints (`mcmc.sh` train command) unless a checkpoint exists |
| 6 | current-main compression run (`mcmc.sh` eval command) + **sanity gate** vs the repo CSV |
| 7 | quantization sweep (sort + k-means cached per checkpoint) |
| 8 | decision rule + RD plot |
| 9 | Phase 4 fallback sweep (tile 128, per-tile scale + global offset), only if step 8 finds no win |

In [ ]:
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

FORK_URL = "https://github.com/Daceyyreal/gsplat.git"
BRANCH = "feat/png-tile-quantization"
SCENES = ["garden", "bicycle"]
CAP_MAX = 1_000_000  # mcmc.sh "1M GSs"
RESULT_NAME = "benchmark_mcmc_1M_png_compression"

WORK = "/kaggle/working"
SRC_DIR = "/tmp/gsplat"
DATA_ROOT = "/tmp/data/360_v2"
RESULT_DIR = f"{WORK}/results/{RESULT_NAME}"  # checkpoints + current-main compression runs
OUT_DIR = f"{WORK}/tilequant"  # sweep cache, CSV, plots, logs
CSV_PATH = f"{OUT_DIR}/tilequant_results.csv"
WHEEL_ROOT = f"{WORK}/wheels"
MIPNERF360_ZIP = "https://storage.googleapis.com/gresearch/refraw360/360_v2.zip"

MAX_JOBS = "2"  # higher values OOM when building gsplat on Kaggle
SWEEP_TIME_BUDGET_MIN = None  # per scene; baseline and tile 16 always run
SIZE_TOLERANCE = 0.15  # sanity gate: per-scene zip size vs the repo CSV size
MAX_PSNR_DROP_DB = 1.0  # sanity gate: val -> compressed PSNR drop
PY = sys.executable


def data_factor(scene):  # as in mcmc.sh
    return 2 if scene in ("bonsai", "counter", "kitchen", "room") else 4


def sh(cmd, cwd=None, env=None, log=None):
    """Run a shell command and fail loudly. With `log`, output goes to that file."""
    print(f"$ {cmd}", flush=True)
    full_env = {**os.environ, **(env or {})}
    if log is None:
        subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=full_env)
        return
    with open(log, "a") as f:
        proc = subprocess.run(cmd, shell=True, cwd=cwd, env=full_env, stdout=f, stderr=subprocess.STDOUT)
    if proc.returncode != 0:
        with open(log) as f:
            print(f.read()[-6000:])
        raise subprocess.CalledProcessError(proc.returncode, cmd)


def record_timing(name, seconds):
    path = f"{OUT_DIR}/timings.json"
    timings = json.load(open(path)) if os.path.exists(path) else {}
    timings[name] = seconds
    json.dump(timings, open(path, "w"), indent=2)


# Restore the output of a previous session of this notebook, if attached as input.
for sub, marker in (("results", RESULT_NAME), ("tilequant", "sweep"), ("wheels", "")):
    for src in glob.glob(f"/kaggle/input/*/{sub}") + glob.glob(f"/kaggle/input/*/*/{sub}"):
        if marker and not os.path.exists(f"{src}/{marker}"):
            continue
        print(f"restoring {src} -> {WORK}/{sub}")
        os.makedirs(f"{WORK}/{sub}", exist_ok=True)
        sh(f"cp -rn {src}/. {WORK}/{sub}/")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
if os.path.isdir("/usr/local/cuda/bin"):
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
    os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")

sh("nvidia-smi")
sh("nvcc --version | tail -n 2 || echo 'nvcc not found'")
sh(f"{PY} --version")
for path in (WORK, "/tmp"):
    usage = shutil.disk_usage(path)
    print(f"{path}: {usage.free / 1e9:.1f} GB free of {usage.total / 1e9:.1f} GB")
print("zip:", shutil.which("zip"))

In [ ]:
def torch_info():
    out = subprocess.check_output(
        [PY, "-c", "import json, torch; print(json.dumps({'version': torch.__version__, "
         "'cuda': torch.version.cuda, 'cap': '.'.join(map(str, torch.cuda.get_device_capability(0)))}))"],
        text=True,
    )
    return json.loads(out.strip().splitlines()[-1])


# 3a. Fork branch.
if not os.path.isdir(f"{SRC_DIR}/.git"):
    sh(f"git clone --recursive --branch {BRANCH} {FORK_URL} {SRC_DIR}")
else:
    sh(f"git -C {SRC_DIR} fetch origin {BRANCH} && git -C {SRC_DIR} checkout -q FETCH_HEAD "
       f"&& git -C {SRC_DIR} submodule update --init --recursive")
COMMIT = subprocess.check_output(["git", "-C", SRC_DIR, "rev-parse", "HEAD"], text=True).strip()
print("gsplat commit:", COMMIT)

# 3b. Torch: keep Kaggle's build if gsplat supports it (>= 2.7), else the version pinned in examples/.
TORCH = torch_info()
if tuple(int(x) for x in TORCH["version"].split("+")[0].split(".")[:2]) < (2, 7):
    sh(f"{PY} -m pip install -q torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cu126")
    TORCH = torch_info()
print("torch:", TORCH)
os.environ["TORCH_CUDA_ARCH_LIST"] = TORCH["cap"]
os.environ["MAX_JOBS"] = MAX_JOBS
with open("/tmp/torch_constraint.txt", "w") as f:
    f.write(f"torch=={TORCH['version']}\n")
PIP = f"{PY} -m pip install -q -c /tmp/torch_constraint.txt"

t0 = time.time()
# 3c. Example dependencies from examples/requirements.txt, without the torch pins and the
# extensions mcmc.sh does not use. CUDA extensions are built without build isolation.
req_lines = [l.strip() for l in open(f"{SRC_DIR}/examples/requirements.txt")]
req_lines = [l for l in req_lines if l and not l.startswith("#")]
skip = ("torch==", "torchvision==", "nvidia-ncore", "ppisp", "fused-bilagrid", "fused-ssim")
with open("/tmp/requirements_bench.txt", "w") as f:
    f.write("\n".join(l for l in req_lines if not any(s in l for s in skip)) + "\n")
sh(f"{PIP} -r /tmp/requirements_bench.txt remotezip pandas")
fused_ssim = next(l for l in req_lines if "fused-ssim" in l)
sh(f"{PIP} --no-build-isolation {fused_ssim}")
# PngCompression extras (see the PngCompression docstring / setup.py dev extras).
sh(f"{PIP} --no-build-isolation git+https://github.com/fraunhoferhhi/PLAS.git")
sh(f"{PIP} torchpq cupy-cuda{TORCH['cuda'].split('.')[0]}x")

# 3d. gsplat wheel, cached by the gsplat/ source tree, setup.py, torch version and GPU arch.
tree = subprocess.check_output(["git", "-C", SRC_DIR, "rev-parse", "HEAD:gsplat"], text=True).strip()
setup_rev = subprocess.check_output(["git", "-C", SRC_DIR, "rev-parse", "HEAD:setup.py"], text=True).strip()
wheel_key = f"{tree[:12]}-{setup_rev[:8]}-torch{TORCH['version'].replace('+', '_')}-sm{TORCH['cap']}"
wheel_dir = f"{WHEEL_ROOT}/{wheel_key}"
if not glob.glob(f"{wheel_dir}/gsplat-*.whl"):
    os.makedirs(wheel_dir, exist_ok=True)
    tb = time.time()
    sh(f"{PY} -m pip wheel -v --no-build-isolation --no-deps -w {wheel_dir} {SRC_DIR}",
       log=f"{OUT_DIR}/gsplat_build.log")
    record_timing("gsplat_wheel_build_s", time.time() - tb)
wheel = glob.glob(f"{wheel_dir}/gsplat-*.whl")[0]
sh(f"{PY} -m pip uninstall -y -q gsplat || true")
sh(f"{PIP} {wheel}")
record_timing("install_s", time.time() - t0)

smoke = """
import inspect, torch, gsplat
from gsplat import rasterization
from gsplat.compression import PngCompression
assert "tile_size" in inspect.signature(PngCompression).parameters, "not the tile-quantization branch"
n, dev = 100, "cuda"
means = torch.randn(n, 3, device=dev) + torch.tensor([0.0, 0.0, 5.0], device=dev)
quats = torch.randn(n, 4, device=dev)
scales = torch.rand(n, 3, device=dev) * 0.1
opacities = torch.rand(n, device=dev)
colors = torch.rand(n, 3, device=dev)
Ks = torch.tensor([[[100.0, 0.0, 50.0], [0.0, 100.0, 50.0], [0.0, 0.0, 1.0]]], device=dev)
img, _, _ = rasterization(means, quats, scales, opacities, colors, torch.eye(4, device=dev)[None], Ks, 100, 100)
print("gsplat", gsplat.__version__, "render ok", tuple(img.shape))
"""
open("/tmp/gsplat_smoke.py", "w").write(smoke)
sh(f"{PY} /tmp/gsplat_smoke.py", cwd="/tmp")

In [ ]:
def find_input_scene(scene, max_depth=5):
    """A COLMAP scene folder named `scene` under /kaggle/input (e.g. an existing 360_v2 dataset)."""
    for root, dirs, _ in os.walk("/kaggle/input", followlinks=True):
        if os.path.basename(root) == scene and "sparse" in dirs and "images" in dirs:
            return root
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        dirs[:] = [] if depth >= max_depth else [
            d for d in dirs if not d.startswith("images") and d not in ("ckpts", "renders", "compression", "wheels")
        ]
    return None


def link_scene(src, dst):
    """Writable copy of a read-only scene: symlinks, except downscaled PNG folders that the
    parser may add files to (it writes images_<factor>_png next to images/)."""
    os.makedirs(dst, exist_ok=True)
    for name in os.listdir(src):
        target = os.path.join(dst, name)
        if os.path.lexists(target):
            continue
        if name.endswith("_png"):
            shutil.copytree(os.path.join(src, name), target)
        else:
            os.symlink(os.path.join(src, name), target)


def download_scene(scene, dst):
    """Only the files simple_trainer needs, read from 360_v2.zip with HTTP range requests."""
    from remotezip import RemoteZip

    wanted = re.compile(rf"^(?:.*/)?{scene}/((?:images|images_{data_factor(scene)}|sparse)/.+|poses_bounds\.npy)$")
    with RemoteZip(MIPNERF360_ZIP) as z:
        members = [(m, wanted.match(m.filename)) for m in z.infolist() if not m.is_dir()]
        members = [(m, match.group(1)) for m, match in members if match]
        total = sum(m.file_size for m, _ in members)
        free = shutil.disk_usage("/tmp").free
        print(f"{scene}: {len(members)} files, {total / 1e9:.2f} GB; /tmp free {free / 1e9:.1f} GB", flush=True)
        if total * 1.5 > free:
            raise RuntimeError(f"not enough space in /tmp for {scene}")
        for m, rel in members:
            out = os.path.join(dst, rel)
            if os.path.exists(out) and os.path.getsize(out) == m.file_size:
                continue
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with z.open(m) as fsrc, open(out + ".part", "wb") as fdst:
                shutil.copyfileobj(fsrc, fdst, 16 << 20)
            os.replace(out + ".part", out)


SCENE_DIRS = {}
t0 = time.time()
for scene in SCENES:
    dst = f"{DATA_ROOT}/{scene}"
    src = find_input_scene(scene)
    if src is not None:
        print(f"{scene}: using Kaggle input {src}")
        link_scene(src, dst)
    else:
        download_scene(scene, dst)
    n_images = len(os.listdir(f"{dst}/images"))
    print(f"{scene}: {dst} ({n_images} images, {sorted(os.listdir(dst))})")
    SCENE_DIRS[scene] = dst
record_timing("data_s", time.time() - t0)
sh("df -h /tmp /kaggle/working")

In [ ]:
CKPTS = {}
for scene in SCENES:
    scene_result = f"{RESULT_DIR}/{scene}"
    ckpt = f"{scene_result}/ckpts/ckpt_29999_rank0.pt"
    if os.path.exists(ckpt):
        print(f"{scene}: checkpoint exists, skipping training ({ckpt})")
        CKPTS[scene] = ckpt
        continue
    os.makedirs(scene_result, exist_ok=True)
    t0 = time.time()
    # train without eval (benchmarks/compression/mcmc.sh)
    sh(
        f"CUDA_VISIBLE_DEVICES=0 {PY} simple_trainer.py mcmc --eval_steps -1 --disable_viewer "
        f"--data_factor {data_factor(scene)} --strategy.cap-max {CAP_MAX} "
        f"--data_dir {SCENE_DIRS[scene]}/ --result_dir {scene_result}/",
        cwd=f"{SRC_DIR}/examples",
        log=f"{OUT_DIR}/train_{scene}.log",
    )
    record_timing(f"train_{scene}_s", time.time() - t0)
    assert os.path.exists(ckpt), f"training did not write {ckpt}"
    # Only the final checkpoint is used; drop the 7k one to stay within /kaggle/working limits.
    for extra in glob.glob(f"{scene_result}/ckpts/ckpt_*_rank0.pt"):
        if extra != ckpt:
            os.remove(extra)
    CKPTS[scene] = ckpt
    print(f"{scene}: trained in {(time.time() - t0) / 60:.1f} min")
    for stats in sorted(glob.glob(f"{scene_result}/stats/train_step*_rank0.json"))[-1:]:
        print(open(stats).read())

In [ ]:
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, f"{SRC_DIR}/kaggle")
import tilequant_analysis as ta
import tilequant_sweep as ts

if shutil.which("zip") is None:
    sh("apt-get install -y -q zip || (apt-get update -q && apt-get install -y -q zip)")

for scene in SCENES:
    scene_result = f"{RESULT_DIR}/{scene}"
    if os.path.exists(f"{scene_result}/stats/compress_step29999.json"):
        print(f"{scene}: current-main compression run exists, skipping")
        continue
    t0 = time.time()
    # eval: use vgg for lpips to align with other benchmarks (benchmarks/compression/mcmc.sh)
    sh(
        f"CUDA_VISIBLE_DEVICES=0 {PY} simple_trainer.py mcmc --disable_viewer "
        f"--data_factor {data_factor(scene)} --strategy.cap-max {CAP_MAX} "
        f"--data_dir {SCENE_DIRS[scene]}/ --result_dir {scene_result}/ "
        f"--lpips_net vgg --compression png --ckpt {CKPTS[scene]}",
        cwd=f"{SRC_DIR}/examples",
        log=f"{OUT_DIR}/main_compression_{scene}.log",
    )
    record_timing(f"main_compression_{scene}_s", time.time() - t0)

# Zip + summary exactly as mcmc.sh does (writes <scene>/compression.zip and compress_summary.json).
sh(f"{PY} benchmarks/compression/summarize_stats.py --results_dir {RESULT_DIR} --scenes {' '.join(SCENES)}",
   cwd=f"{SRC_DIR}/examples")

repo_row = ta.read_repo_row(f"{SRC_DIR}/examples/benchmarks/compression/results/MipNeRF360.csv", CAP_MAX)
scene_stats = {s: ta.canonical_scene_stats(f"{RESULT_DIR}/{s}") for s in SCENES}
gate_table, gate_failures = ta.sanity_gate(scene_stats, repo_row, CAP_MAX, SIZE_TOLERANCE, MAX_PSNR_DROP_DB)
display(gate_table)

logged = set()
if os.path.exists(CSV_PATH):
    prev = pd.read_csv(CSV_PATH)
    logged = set(prev.loc[prev["Submethod"] == "main_cli", "scene"])
for scene, s in scene_stats.items():
    if scene in logged:
        continue
    ts.append_row(CSV_PATH, {
        "Submethod": "main_cli", "PSNR": s["psnr"], "SSIM": s["ssim"], "LPIPS": s["lpips"],
        "Size [Bytes]": s["zip_bytes"], "#Gaussians": s["num_GS"], "scene": scene,
        "variant": "main_cli", "tile_size": "", "bits_means": 16, "bits_other": 8,
        "size_bytes": s["size_bytes"], "zip_bytes": s["zip_bytes"],
        "gsplat_commit": COMMIT[:12], "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    })

with open(f"{OUT_DIR}/sanity_gate.json", "w") as f:
    json.dump({"table": gate_table.to_dict("records"), "failures": gate_failures,
               "repo_row": repo_row, "size_tolerance": SIZE_TOLERANCE,
               "max_psnr_drop_db": MAX_PSNR_DROP_DB}, f, indent=2, default=float)
print("Note: the repo CSV only has the mean over 9 MipNeRF360 scenes, so per-scene PSNR is checked "
      "as the val -> compressed drop; the 2-scene mean above is for reference only.")
if gate_failures:
    raise RuntimeError("SANITY GATE FAILED - current-main baseline does not match the repo CSV; "
                       "check the settings before trusting the sweep:\n" + "\n".join(gate_failures))
print("Sanity gate passed.")

In [ ]:
def run_sweep(grid):
    for scene in SCENES:
        cmd = [
            PY, f"{SRC_DIR}/kaggle/tilequant_sweep.py", "--scene", scene,
            "--data_dir", SCENE_DIRS[scene], "--ckpt", CKPTS[scene],
            "--work_dir", f"{OUT_DIR}/sweep/{scene}", "--csv", CSV_PATH, "--grid", grid,
            "--data_factor", str(data_factor(scene)), "--cap_max", str(CAP_MAX), "--commit", COMMIT[:12],
        ]
        if SWEEP_TIME_BUDGET_MIN is not None:
            cmd += ["--time_budget_min", str(SWEEP_TIME_BUDGET_MIN)]
        log = f"{OUT_DIR}/sweep_{grid}_{scene}.log"
        print("$", " ".join(cmd), flush=True)
        t0 = time.time()
        with open(log, "a") as f:
            proc = subprocess.Popen(
                cmd, cwd=f"{SRC_DIR}/examples", env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            )
            for line in proc.stdout:
                f.write(line)
                if line.startswith(f"[{scene}]") or line.startswith("Built cache") or "Error" in line:
                    print(line, end="", flush=True)
            if proc.wait() != 0:
                with open(log) as g:
                    print(g.read()[-6000:])
                raise RuntimeError(f"sweep '{grid}' failed for {scene}, see {log}")
        record_timing(f"sweep_{grid}_{scene}_s", time.time() - t0)


run_sweep("main")

In [ ]:
def report(tag):
    df = pd.read_csv(CSV_PATH)
    decisions = {col: ta.decide(df, SCENES, size_col=col) for col in ("size_bytes", "zip_bytes")}
    for col, d in decisions.items():
        print(f"== size measure: {col}")
        for scene, v in d["per_scene"].items():
            print(f"  {scene}: baseline PSNR {v['baseline_psnr']:.3f} dB, size {v['baseline_size']} B; "
                  f"{len(v['winners'])}/{v['n_candidates']} configs at <= size and >= PSNR; "
                  f"global RD front points matched by a tiled config: "
                  f"{v['global_front_dominated']}/{len(v['global_front'])}")
        print(f"  winners on all scenes: {len(d['common_winners'])}; RD shift on all scenes: "
              f"{d['rd_shift_all_scenes']}; PR-worthy: {d['pr_worthy']}")
        if d["common_winners_table"]:
            display(pd.DataFrame(d["common_winners_table"]).round(4))
    for scene in SCENES:
        s = df[(df["scene"] == scene) & df["variant"].isin(["baseline", "global", "tile", "goffset"])]
        s = s.assign(scheme=s["variant"] + s["tile_size"].fillna(0).astype(int).map(lambda t: f" {t}" if t else ""))
        print(f"PSNR by bits (rows: means, other) and scheme - {scene}")
        display(s.pivot_table(index=["bits_means", "bits_other"], columns="scheme", values="PSNR").round(3))
        print(f"size_bytes by bits and scheme - {scene}")
        display(s.pivot_table(index=["bits_means", "bits_other"], columns="scheme", values="size_bytes").astype("Int64"))
    with open(f"{OUT_DIR}/decision_{tag}.json", "w") as f:
        json.dump(decisions, f, indent=2)
    plot_path = f"{OUT_DIR}/rd_size_vs_psnr_{tag}.png"
    ta.plot_rd(df, SCENES, plot_path)
    display(Image(plot_path))
    return decisions


DECISION = report("main")
PR_WORTHY = DECISION["size_bytes"]["pr_worthy"]
print("PR-worthy after main sweep:", PR_WORTHY)

In [ ]:
if PR_WORTHY:
    print("A tiled configuration beats the current-main baseline on every scene: skipping the Phase 4 "
          "fallback. Next step: extend SCENES to all MipNeRF360 scenes + Tanks&Temples (mcmc_tt.sh).")
else:
    print("No win in the main sweep: running the Phase 4 fallback grid (tile 128, per-tile scale + global offset).")
    run_sweep("fallback")
    DECISION = report("with_fallback")
    PR_WORTHY = DECISION["size_bytes"]["pr_worthy"]
    print("PR-worthy after fallback:", PR_WORTHY)

In [ ]:
print(json.dumps(json.load(open(f"{OUT_DIR}/timings.json")), indent=2))
sh(f"ls -la {OUT_DIR}")
sh(f"du -sh {WORK}/* || true")